In [7]:
!pip install pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ----------------------------------- ---- 9.4/10.6 MB 52.3 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 44.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pygame
import sys
import time

# ===============================
# CONFIG
# ===============================
WIDTH, HEIGHT = 900, 650
GRID = 25

WHITE = (255,255,255)
GRAY = (200,200,200)
RED = (255,0,0)
GREEN = (0,200,0)
BLUE = (0,0,255)
BLACK = (0,0,0)

pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Smart Graphics Debugger")
clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 24)

# ===============================
# STEP MANAGER
# ===============================
class StepManager:
    def __init__(self):
        self.steps = []
        self.current = 0

    def load(self, steps):
        self.steps = steps
        self.current = 0

    def next(self):
        if self.current < len(self.steps)-1:
            self.current += 1

    def prev(self):
        if self.current > 0:
            self.current -= 1

    def reset(self):
        self.current = 0

    def get(self):
        return self.steps[self.current] if self.steps else None

manager = StepManager()

# ===============================
# ALGORITHMS
# ===============================

def bresenham(x1,y1,x2,y2):
    steps=[]
    dx,dy=x2-x1,y2-y1
    x,y=x1,y1
    p=2*dy-dx
    for _ in range(dx+1):
        steps.append({"x":x,"y":y,"p":p})
        if p>=0:
            y+=1
            p+=2*dy-2*dx
        else:
            p+=2*dy
        x+=1
    return steps

def dda(x1,y1,x2,y2):
    steps=[]
    dx,dy=x2-x1,y2-y1
    n=max(abs(dx),abs(dy))
    xi,yi=dx/n,dy/n
    x,y=x1,y1
    for _ in range(n+1):
        steps.append({"x":round(x),"y":round(y),"rx":x,"ry":y})
        x+=xi;y+=yi
    return steps

def midpoint(cx,cy,r):
    steps=[]
    x,y=0,r
    p=1-r
    while x<=y:
        pts=[(cx+x,cy+y),(cx-x,cy+y),(cx+x,cy-y),(cx-x,cy-y),
             (cx+y,cy+x),(cx-y,cy+x),(cx+y,cy-x),(cx-y,cy-x)]
        steps.append({"points":pts,"p":p})
        if p<0: p+=2*x+3
        else:
            p+=2*(x-y)+5
            y-=1
        x+=1
    return steps

INSIDE,LEFT,RIGHT,BOTTOM,TOP=0,1,2,4,8

def code(x,y,xmin,ymin,xmax,ymax):
    c=INSIDE
    if x<xmin:c|=LEFT
    elif x>xmax:c|=RIGHT
    if y<ymin:c|=BOTTOM
    elif y>ymax:c|=TOP
    return c

def cohen(x1,y1,x2,y2,xmin,ymin,xmax,ymax):
    steps=[]
    while True:
        c1,c2=code(x1,y1,xmin,ymin,xmax,ymax),code(x2,y2,xmin,ymin,xmax,ymax)
        steps.append({"x1":x1,"y1":y1,"x2":x2,"y2":y2,
                      "c1":format(c1,'04b'),"c2":format(c2,'04b')})
        if c1==0 and c2==0: break
        if c1 & c2: break

        c_out=c1 if c1 else c2

        if c_out & TOP:
            x=x1+(x2-x1)*(15-y1)/(y2-y1);y=15
        elif c_out & BOTTOM:
            x=x1+(x2-x1)*(5-y1)/(y2-y1);y=5
        elif c_out & RIGHT:
            y=y1+(y2-y1)*(18-x1)/(x2-x1);x=18
        else:
            y=y1+(y2-y1)*(5-x1)/(x2-x1);x=5

        if c_out==c1: x1,y1=x,y
        else: x2,y2=x,y
    return steps

def scanline(poly):
    steps=[]
    ymin=min(y for x,y in poly)
    ymax=max(y for x,y in poly)
    for y in range(ymin,ymax+1):
        inter=[]
        for i in range(len(poly)):
            x1,y1=poly[i];x2,y2=poly[(i+1)%len(poly)]
            if y1<y2: xl,yl,xh,yh=x1,y1,x2,y2
            else: xl,yl,xh,yh=x2,y2,x1,y1
            if yl<=y<yh:
                x=xl+(y-yl)*(xh-xl)/(yh-yl)
                inter.append(x)
        inter.sort()
        steps.append({"y":y,"xs":inter})
    return steps

# ===============================
# DRAW
# ===============================

def grid():
    for x in range(0,WIDTH,GRID):
        pygame.draw.line(screen,GRAY,(x,0),(x,HEIGHT))
    for y in range(0,HEIGHT,GRID):
        pygame.draw.line(screen,GRAY,(0,y),(WIDTH,y))

def pixel(x,y,color):
    pygame.draw.rect(screen,color,(x*GRID,y*GRID,GRID,GRID))

def draw():
    steps=manager.steps
    i=manager.current

    if not steps: return

    for idx,s in enumerate(steps):
        color=GREEN if idx<i else RED if idx==i else GRAY

        if "x" in s:
            pixel(s["x"],s["y"],color)

        elif "points" in s:
            for p in s["points"]:
                pixel(p[0],p[1],color)

        elif "x1" in s:
            pygame.draw.rect(screen,BLUE,(5*GRID,5*GRID,13*GRID,10*GRID),2)
            pygame.draw.line(screen,color,
                             (s["x1"]*GRID,s["y1"]*GRID),
                             (s["x2"]*GRID,s["y2"]*GRID),3)

        elif "xs" in s:
            y=s["y"]
            for j in range(0,len(s["xs"]),2):
                if j+1<len(s["xs"]):
                    pygame.draw.line(screen,color,
                        (s["xs"][j]*GRID,y*GRID),
                        (s["xs"][j+1]*GRID,y*GRID),3)

# ===============================
# LOAD DEFAULT
# ===============================
algo="Bresenham"
manager.load(bresenham(2,2,20,10))
auto=False

# ===============================
# MAIN LOOP
# ===============================
while True:
    screen.fill(WHITE)
    grid()
    draw()

    cur=manager.get()
    if cur:
        text=str(cur)
        img=font.render(text,True,BLACK)
        screen.blit(img,(10,10))

    for e in pygame.event.get():
        if e.type==pygame.QUIT:
            pygame.quit()
            sys.exit()

        if e.type==pygame.KEYDOWN:
            if e.key==pygame.K_RIGHT: manager.next()
            if e.key==pygame.K_LEFT: manager.prev()
            if e.key==pygame.K_r: manager.reset()
            if e.key==pygame.K_SPACE: auto=not auto

            if e.key==pygame.K_1:
                algo="Bresenham"; manager.load(bresenham(2,2,20,10))
            if e.key==pygame.K_2:
                algo="DDA"; manager.load(dda(2,2,20,10))
            if e.key==pygame.K_3:
                algo="Circle"; manager.load(midpoint(10,10,8))
            if e.key==pygame.K_4:
                algo="Clipping"; manager.load(cohen(2,2,22,18,5,5,18,15))
            if e.key==pygame.K_5:
                algo="Scanline"; manager.load(scanline([(5,5),(15,5),(18,10),(10,15),(5,10)]))

    if auto:
        manager.next()
        time.sleep(0.2)

    pygame.display.flip()
    clock.tick(60)